# Node Diffusion 注意力结构消融评估
对比三种注意力变体（**AdjAttn only** / **GlobalAttn only** / **Dual-stream**）在相同 100 条训练集样本上的 Coord-RMSE，并逐样本可视化对比。

推理方式：**DDPM 标准 1000 步逆采样**（非 DDIM 加速）。

In [ ]:
import os, shutil
REPO_DIR = '/kaggle/working/PlanDiffusion_wzm'
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
os.system(f'git clone https://github.com/WeeZHnMin/PlanDiffusion_wzm.git {REPO_DIR}')
print(f'repo ready: {REPO_DIR}')

In [ ]:
import os

BERT_PATH = 'bert-base-uncased'
DATA_PATH = '/kaggle/input/datasets/yahiie/node-diffusion-6k/graph_dataset_6k.npz'
CKPT_DIR  = '/kaggle/working/checkpoints_eval'
VIZ_DIR   = '/kaggle/working/viz_ablation'
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(VIZ_DIR,  exist_ok=True)

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')

HF_REPOS = {
    'adj_only':    'wzmmmm/plandiff-adj-cross-6k',
    'global_only': 'wzmmmm/plandiff-global-cross-6k',
    'dual_stream': 'wzmmmm/plandiff-double-cross-6k',
}

MODEL_CHANNELS = 384
NUM_LAYERS     = 6
NUM_HEADS      = 6
TIMESTEPS      = 1000
N_EVAL         = 100
N_VIZ          = 8
SEED           = 42

print('config OK')
hf_ok = ('OK (' + HF_TOKEN[:8] + '...)') if HF_TOKEN else '未设置'
print(f'HF_TOKEN: {hf_ok}')

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import BertModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')

# ── 共用组件 ──────────────────────────────────────────────────────────────────
def timestep_embedding(timesteps, dim):
    half  = dim // 2
    freqs = torch.exp(
        -math.log(10000) * torch.arange(half, dtype=torch.float32, device=timesteps.device) / half
    )
    args = timesteps[:, None].float() * freqs[None]
    return torch.cat([torch.cos(args), torch.sin(args)], dim=-1)

def attention(q, k, v, d_k, mask=None, dropout=None):
    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask.unsqueeze(1) == 1, -1e4)
    scores = F.softmax(scores.float(), dim=-1).to(q.dtype)
    if dropout is not None:
        scores = dropout(scores)
    return torch.matmul(scores, v)

class MultiHeadAttention(nn.Module):
    def __init__(self, heads, d_model, dropout=0.1):
        super().__init__()
        self.d_k = d_model // heads
        self.h   = heads
        self.q_linear = nn.Linear(d_model, d_model)
        self.k_linear = nn.Linear(d_model, d_model)
        self.v_linear = nn.Linear(d_model, d_model)
        self.out      = nn.Linear(d_model, d_model)
        self.dropout  = nn.Dropout(dropout)
    def forward(self, q, k, v, mask=None):
        bs = q.size(0)
        q  = self.q_linear(q).view(bs, -1, self.h, self.d_k).transpose(1, 2)
        k  = self.k_linear(k).view(bs, -1, self.h, self.d_k).transpose(1, 2)
        v  = self.v_linear(v).view(bs, -1, self.h, self.d_k).transpose(1, 2)
        out = attention(q, k, v, self.d_k, mask, self.dropout)
        out = out.transpose(1, 2).contiguous().view(bs, -1, self.h * self.d_k)
        return self.out(out)

class FeedForward(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_model * 2)
        self.linear2 = nn.Linear(d_model * 2, d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        return self.linear2(self.dropout(F.relu(self.linear1(x))))

# ── 变体 1：Dual-stream (AdjAttn + GlobalAttn + CrossAttn) ────────────────────
class EncoderLayer_Dual(nn.Module):
    def __init__(self, d_model, heads, dropout=0.1):
        super().__init__()
        self.norm1       = nn.LayerNorm(d_model)
        self.norm_cross  = nn.LayerNorm(d_model)
        self.norm2       = nn.LayerNorm(d_model)
        self.adj_attn    = MultiHeadAttention(heads, d_model, dropout)
        self.global_attn = MultiHeadAttention(heads, d_model, dropout)
        self.cross_attn  = MultiHeadAttention(heads, d_model, dropout)
        self.ff          = FeedForward(d_model, dropout)
        self.dropout     = nn.Dropout(dropout)
    def forward(self, x, adj_mask, text_feat, text_mask):
        x2 = self.norm1(x)
        x  = x + self.dropout(
            self.adj_attn(x2, x2, x2, adj_mask) + self.global_attn(x2, x2, x2, None)
        )
        x2 = self.norm_cross(x)
        x  = x + self.dropout(self.cross_attn(x2, text_feat, text_feat, text_mask))
        x2 = self.norm2(x)
        x  = x + self.dropout(self.ff(x2))
        return x

# ── 变体 2：AdjAttn only (AdjAttn + CrossAttn) ───────────────────────────────
class EncoderLayer_Adj(nn.Module):
    def __init__(self, d_model, heads, dropout=0.1):
        super().__init__()
        self.norm1      = nn.LayerNorm(d_model)
        self.norm_cross = nn.LayerNorm(d_model)
        self.norm2      = nn.LayerNorm(d_model)
        self.adj_attn   = MultiHeadAttention(heads, d_model, dropout)
        self.cross_attn = MultiHeadAttention(heads, d_model, dropout)
        self.ff         = FeedForward(d_model, dropout)
        self.dropout    = nn.Dropout(dropout)
    def forward(self, x, adj_mask, text_feat, text_mask):
        x2 = self.norm1(x)
        x  = x + self.dropout(self.adj_attn(x2, x2, x2, adj_mask))
        x2 = self.norm_cross(x)
        x  = x + self.dropout(self.cross_attn(x2, text_feat, text_feat, text_mask))
        x2 = self.norm2(x)
        x  = x + self.dropout(self.ff(x2))
        return x

# ── 变体 3：GlobalAttn only (GlobalAttn + CrossAttn) ─────────────────────────
class EncoderLayer_Global(nn.Module):
    def __init__(self, d_model, heads, dropout=0.1):
        super().__init__()
        self.norm1       = nn.LayerNorm(d_model)
        self.norm_cross  = nn.LayerNorm(d_model)
        self.norm2       = nn.LayerNorm(d_model)
        self.global_attn = MultiHeadAttention(heads, d_model, dropout)
        self.cross_attn  = MultiHeadAttention(heads, d_model, dropout)
        self.ff          = FeedForward(d_model, dropout)
        self.dropout     = nn.Dropout(dropout)
    def forward(self, x, adj_mask, text_feat, text_mask):
        x2 = self.norm1(x)
        x  = x + self.dropout(self.global_attn(x2, x2, x2, None))
        x2 = self.norm_cross(x)
        x  = x + self.dropout(self.cross_attn(x2, text_feat, text_feat, text_mask))
        x2 = self.norm2(x)
        x  = x + self.dropout(self.ff(x2))
        return x

# ── 通用 Transformer 骨架（接受不同 EncoderLayer）─────────────────────────────
class NodeDiffusionTransformer(nn.Module):
    def __init__(self, layer_cls, model_channels=384, num_layers=6, num_heads=6,
                 dropout=0.1, bert_name='bert-base-uncased'):
        super().__init__()
        self.model_channels = model_channels
        self.time_embed = nn.Sequential(
            nn.Linear(model_channels, model_channels),
            nn.SiLU(),
            nn.Linear(model_channels, model_channels),
        )
        self.input_emb = nn.Linear(2, model_channels)
        self.bert = BertModel.from_pretrained(bert_name)
        for p in self.bert.parameters():
            p.requires_grad = False
        self.text_proj = nn.Linear(self.bert.config.hidden_size, model_channels)
        self.layers = nn.ModuleList(
            [layer_cls(model_channels, num_heads, dropout) for _ in range(num_layers)]
        )
        self.coord_head = nn.Sequential(
            nn.Linear(model_channels, model_channels),
            nn.ReLU(),
            nn.Linear(model_channels, model_channels // 2),
            nn.Linear(model_channels // 2, 2),
        )
    def _build_adj_mask(self, adj_matrix, node_mask):
        adj_mask = 1 - adj_matrix
        pad_keys = (1 - node_mask).unsqueeze(1)
        return torch.clamp(adj_mask + pad_keys, 0, 1)
    def forward(self, x, timesteps, adj_matrix, node_mask,
                prompt_tokens=None, prompt_mask=None, **kwargs):
        del kwargs
        B, _, N = x.shape
        x        = x.permute(0, 2, 1).float()
        t_emb    = self.time_embed(timestep_embedding(timesteps, self.model_channels)).unsqueeze(1)
        node_emb = self.input_emb(x) + t_emb
        adj_mask = self._build_adj_mask(adj_matrix.float(), node_mask.float())
        if prompt_tokens is not None:
            bert_attn = prompt_mask if prompt_mask is not None else (prompt_tokens != 0).long()
            with torch.no_grad():
                text_hidden = self.bert(
                    input_ids=prompt_tokens,
                    attention_mask=bert_attn
                ).last_hidden_state
            text_feat = self.text_proj(text_hidden)
            text_mask = (1 - bert_attn.float()).unsqueeze(1)
        else:
            text_feat = torch.zeros(B, 1, self.model_channels,
                                    device=node_emb.device, dtype=node_emb.dtype)
            text_mask = None
        seq = node_emb
        for layer in self.layers:
            seq = layer(seq, adj_mask, text_feat, text_mask)
        return self.coord_head(seq).permute(0, 2, 1)

print('模型类定义完成')

In [ ]:
import math
import torch

class GaussianDiffusion:
    # Cosine noise schedule (Nichol & Dhariwal 2021)
    def __init__(self, timesteps=1000):
        self.T = timesteps
        t          = torch.arange(timesteps + 1) / timesteps
        f          = torch.cos((t + 0.008) / 1.008 * math.pi / 2) ** 2
        alphas_bar = f / f[0]
        betas      = (1 - alphas_bar[1:] / alphas_bar[:-1]).clamp(max=0.999)
        alphas_bar = alphas_bar[1:]
        alphas          = 1.0 - betas
        alphas_bar_prev = torch.cat([torch.tensor([1.0]), alphas_bar[:-1]])
        self.betas              = betas
        self.alphas             = alphas
        self.alphas_bar         = alphas_bar
        self.alphas_bar_prev    = alphas_bar_prev
        self.posterior_variance = (
            betas * (1 - alphas_bar_prev) / (1 - alphas_bar)
        ).clamp(min=1e-20)
    def _to(self, device):
        for attr in ['betas', 'alphas', 'alphas_bar', 'alphas_bar_prev', 'posterior_variance']:
            setattr(self, attr, getattr(self, attr).to(device))
        return self

@torch.no_grad()
def ddpm_sample(model, diffusion, cond, device):
    # DDPM 标准 1000 步逆采样
    diffusion._to(device)
    x_t = torch.randn(1, 2, 40, device=device)
    for t_val in reversed(range(diffusion.T)):
        t_batch = torch.tensor([t_val], device=device, dtype=torch.long)
        eps     = model(x_t, t_batch, **cond).float()
        ab_t    = diffusion.alphas_bar[t_val]
        ab_prev = diffusion.alphas_bar_prev[t_val]
        alpha_t = diffusion.alphas[t_val]
        beta_t  = diffusion.betas[t_val]
        # 预测 x0
        x0_pred = (x_t - (1 - ab_t).sqrt() * eps) / ab_t.sqrt().clamp(min=1e-3)
        x0_pred = x0_pred.clamp(-300, 300)
        # 后验均值
        mu = (ab_prev.sqrt() * beta_t / (1 - ab_t)) * x0_pred + \
             (alpha_t.sqrt() * (1 - ab_prev) / (1 - ab_t)) * x_t
        if t_val > 0:
            x_t = mu + diffusion.posterior_variance[t_val].sqrt() * torch.randn_like(x_t)
        else:
            x_t = mu
    return x_t[0].permute(1, 0).cpu().numpy()  # [40, 2]

diffusion = GaussianDiffusion(timesteps=TIMESTEPS)
print('GaussianDiffusion + DDPM sampler 定义完成')

In [ ]:
from huggingface_hub import hf_hub_download
import torch

LAYER_CLS_MAP = {
    'adj_only':    EncoderLayer_Adj,
    'global_only': EncoderLayer_Global,
    'dual_stream': EncoderLayer_Dual,
}

models = {}
for name, repo_id in HF_REPOS.items():
    print(f'正在加载 {name} <- {repo_id} ...')
    ckpt_path = hf_hub_download(
        repo_id=repo_id, filename='latest.pt',
        token=HF_TOKEN,
        local_dir=os.path.join(CKPT_DIR, name),
        force_download=True,
    )
    m = NodeDiffusionTransformer(
        layer_cls=LAYER_CLS_MAP[name],
        model_channels=MODEL_CHANNELS,
        num_layers=NUM_LAYERS,
        num_heads=NUM_HEADS,
        bert_name=BERT_PATH,
    ).to(device)
    ckpt  = torch.load(ckpt_path, map_location=device)
    state = ckpt['model']
    if any(k.startswith('module.') for k in state):
        state = {k[7:]: v for k, v in state.items()}
    m.load_state_dict(state, strict=False)
    m.eval()
    step = ckpt.get('step', '?')
    models[name] = m
    print(f'  {name}: loaded  step={step}')

print('\n所有模型加载完成')

In [ ]:
import numpy as np
import torch

torch.manual_seed(SEED)
np.random.seed(SEED)

data  = np.load(DATA_PATH, allow_pickle=True)
total = len(data['node_coords'])
rng   = np.random.default_rng(SEED)
idxs  = sorted(rng.choice(total, size=N_EVAL, replace=False).tolist())
print(f'数据集共 {total} 条，随机选取 {N_EVAL} 条评估')

rmse_results = {name: [] for name in HF_REPOS}
pred_cache   = {name: [] for name in HF_REPOS}
gt_cache, mask_cache, adj_cache, type_cache = [], [], [], []

for i, idx in enumerate(idxs):
    gt_coords = data['node_coords'][idx].astype('float32')   # [40, 2]
    adj_np    = data['adj_matrix'][idx].astype('float32')    # [40, 40]
    mask_np   = data['node_mask'][idx].astype('float32')     # [40]
    types_np  = data['node_combo_ids'][idx].astype('int64')  # [40]
    ptok_np   = data['prompt_tokens'][idx].astype('int64')   # [128]
    pmask_np  = data['prompt_mask'][idx].astype('float32')   # [128]

    cond = {
        'adj_matrix':    torch.from_numpy(adj_np[None]).to(device),
        'node_mask':     torch.from_numpy(mask_np[None]).to(device),
        'prompt_tokens': torch.from_numpy(ptok_np[None]).to(device),
        'prompt_mask':   torch.from_numpy(pmask_np[None]).to(device),
    }
    valid = mask_np > 0.5

    gt_cache.append(gt_coords)
    adj_cache.append(adj_np)
    mask_cache.append(valid)
    type_cache.append(types_np)

    for name, model in models.items():
        pred_xy = ddpm_sample(model, diffusion, cond, device)   # [40, 2]
        rmse    = float(np.sqrt(np.mean((pred_xy[valid] - gt_coords[valid]) ** 2)))
        rmse_results[name].append(rmse)
        pred_cache[name].append(pred_xy)

    if (i + 1) % 10 == 0:
        row = ' | '.join(f'{n}: {np.mean(rmse_results[n]):.2f}' for n in HF_REPOS)
        print(f'[{i+1}/{N_EVAL}] {row}')

print('\n=== 评估结果 (Coord-RMSE, 像素) ===')
print(f'{"变体":<20} {"均值":>8} {"中位数":>8} {"最优":>8} {"最差":>8}')
print('-' * 52)
for name in HF_REPOS:
    r = rmse_results[name]
    print(f'{name:<20} {np.mean(r):>8.2f} {np.median(r):>8.2f}'
          f' {np.min(r):>8.2f} {np.max(r):>8.2f}')

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import display, Image as IPImage
import numpy as np

COMBO_COLORS = [
    '#A9CDE8','#7BB9E0','#4D9FD5','#1A6FA6',
    '#A8D5A2','#6EBC68','#3A9E35','#1E7B19',
    '#F5D48B','#F0BE45','#E89E10','#B87A00',
    '#D4A8D4','#B87BB8','#8F4F8F','#6B2D6B',
    '#F5A8A8','#E86060','#D02020','#A00000',
    '#BBBBBB','#999999','#777777','#555555',
    '#FFDDB0','#FFB86C','#E88C30','#C06010',
    '#B0E0E0','#70C0C0','#30A0A0','#008080',
]

def type_color(tid):
    idx = (int(tid) - 1) % len(COMBO_COLORS) if 1 <= int(tid) <= 32 else -1
    return COMBO_COLORS[idx] if idx >= 0 else '#DDDDDD'

def draw_graph(ax, xy, types, adj, n, title, rmse=None):
    pts = xy[:n]
    pad = max(10.0, 0.05 * float(np.ptp(pts, axis=0).max()))
    ax.set_xlim(pts[:, 0].min() - pad, pts[:, 0].max() + pad)
    ax.set_ylim(pts[:, 1].min() - pad, pts[:, 1].max() + pad)
    for ii in range(n):
        for jj in range(ii + 1, n):
            if adj[ii, jj] > 0.5:
                ax.plot([xy[ii, 0], xy[jj, 0]], [xy[ii, 1], xy[jj, 1]],
                        color='#AAAAAA', lw=0.8, alpha=0.6, zorder=1)
    for k in range(n):
        c = type_color(types[k])
        ax.scatter(xy[k, 0], xy[k, 1], color=c, s=55, zorder=3,
                   edgecolors='#444444', linewidths=0.5)
    label = title if rmse is None else f'{title}\nRMSE={rmse:.1f}px'
    ax.set_title(label, fontsize=8)
    ax.set_aspect('equal')
    ax.invert_yaxis()
    ax.grid(alpha=0.15)

viz_idxs  = list(range(0, N_EVAL, max(1, N_EVAL // N_VIZ)))[:N_VIZ]
col_names = ['GT', 'AdjAttn only', 'GlobalAttn only', 'Dual-stream']
var_keys  = ['adj_only', 'global_only', 'dual_stream']

for vi, si in enumerate(viz_idxs):
    idx    = idxs[si]
    n_node = int(mask_cache[si].sum())
    gt_xy  = gt_cache[si]
    adj_np = adj_cache[si]
    types  = type_cache[si]

    fig, axes = plt.subplots(1, 4, figsize=(22, 5.5))
    draw_graph(axes[0], gt_xy, types, adj_np, n_node, 'GT')
    for ci, vkey in enumerate(var_keys):
        draw_graph(axes[ci + 1], pred_cache[vkey][si], types, adj_np,
                   n_node, col_names[ci + 1], rmse=rmse_results[vkey][si])

    fig.suptitle(f'sample idx={idx}   n_nodes={n_node}', fontsize=9)
    fig.tight_layout()
    out_path = os.path.join(VIZ_DIR, f'ablation_{vi:02d}_idx{idx}.png')
    fig.savefig(out_path, dpi=130, bbox_inches='tight')
    plt.close(fig)
    display(IPImage(out_path))
    row = '  '.join(f'{k}={rmse_results[k][si]:.1f}' for k in var_keys)
    print(f'sample {idx}: {row}')

print(f'\n{N_VIZ} 张可视化图保存至: {VIZ_DIR}')